# Post-trade report

Turns one run directory written by `backtest` into a post-trade report: PnL and position over
time, fill rate and time to first fill, markouts, queue position against fill outcome, and
risk rejections.

Point it at a run with the `ITCHBT_RUN` environment variable, or edit `RUN_DIR` below:

```sh
ITCHBT_RUN=runs/demo jupyter lab analysis/report.ipynb
```

Prices in the run files are integers in units of 1/10000 USD and timestamps are nanoseconds
since midnight. Everything shown here is converted to dollars, basis points and clock time.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import load
import report

RUN_DIR = Path(os.environ.get("ITCHBT_RUN", "../runs/demo"))
run = load.load_run(RUN_DIR)
meta = run.meta

def hours(ts):
    """Nanoseconds since midnight as fractional hours, for time axes."""
    return ts / 3.6e12

print(f"Run:       {RUN_DIR}")
print(f"Input:     {meta['input']['file']} ({meta['input']['size']:,} bytes)")
print(f"Commit:    {meta['git_commit'][:12]}   Compiler: {meta['compiler']}")
print(f"Strategy:  {meta['config']['strategy']} on {', '.join(run.symbols.values())}")
print(f"Messages:  {meta['stats']['messages']:,} in {meta['timing']['wall_time_s']:.2f} s")
print(f"Orders:    {meta['stats']['orders_submitted']:,} submitted, "
      f"{meta['stats']['orders_rejected_by_risk']:,} rejected by risk, "
      f"{meta['stats']['fills']:,} fills ({meta['stats']['shares_filled']:,} shares)")
print(f"Total PnL: {meta['risk']['total_pnl_x2'] / 20000:,.2f} USD   "
      f"Kill switch: {meta['risk']['kill_switch']}")

## PnL over time

Cash plus position marked at the mid, per symbol and in total. Positions are marked only on
top-of-book records flagged as tradable (session open, symbol not halted), which is the rule
the engine itself uses, so the last point of each line equals the engine's figure in
`run.json`. That equality is checked by the tests, not assumed.

In [ ]:
timelines = report.pnl_timelines(run, step_seconds=1.0)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hours(timelines["ts"]), timelines["total_pnl_usd"], color="black", linewidth=1.2)
ax.axhline(0, color="grey", linewidth=0.5)
ax.set_xlabel("time of day (hours)")
ax.set_ylabel("total PnL (USD)")
ax.set_title("Total PnL")
plt.show()

engine = pd.DataFrame(meta["risk"]["symbols"]).set_index("symbol")
engine["pnl_usd"] = (2 * engine["realised"] + engine["unrealised_x2"]) / 20000
engine["realised_usd"] = engine["realised"] / 10000
engine[["position", "realised_usd", "pnl_usd"]]

In [ ]:
symbols = list(run.symbols.values())
fig, axes = plt.subplots(len(symbols), 1, figsize=(11, 2.6 * len(symbols)), sharex=True,
                         squeeze=False)
for ax, symbol in zip(axes[:, 0], symbols):
    if f"{symbol}_pnl_usd" not in timelines:
        ax.set_title(f"{symbol}: no two-sided market recorded")
        continue
    ax.plot(hours(timelines["ts"]), timelines[f"{symbol}_pnl_usd"], color="tab:blue")
    ax.set_ylabel("PnL (USD)", color="tab:blue")
    ax.axhline(0, color="grey", linewidth=0.5)
    position = ax.twinx()
    position.step(hours(timelines["ts"]), timelines[f"{symbol}_position"], where="post",
                  color="tab:orange", alpha=0.6, linewidth=0.8)
    position.set_ylabel("position (shares)", color="tab:orange")
    ax.set_title(symbol)
axes[-1, 0].set_xlabel("time of day (hours)")
plt.tight_layout()
plt.show()

## Fill rate and time to first fill

Fill rate is over limit orders that went live on the simulated exchange. Time to first fill is
measured from the moment an order went live to its first fill, in simulation time.

In [ ]:
stats = report.fill_stats(run)
print(f"Orders submitted:        {stats['orders_submitted']:,}")
print(f"Orders that went live:   {stats['orders_live']:,}")
print(f"Live orders with a fill: {stats['orders_with_a_fill']:,} "
      f"({stats['fill_rate']:.1%} of live orders, {stats['share_fill_rate']:.1%} of live shares)")

waits = report.time_to_first_fill(run)
if len(waits):
    positive = waits[waits > 0]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    if len(positive):
        ax.hist(positive, bins=pd.Series(positive).quantile([i / 60 for i in range(61)]).unique())
        ax.set_xscale("log")
    ax.set_xlabel("seconds from live to first fill (log scale)")
    ax.set_ylabel("orders")
    ax.set_title(f"Time to first fill ({len(waits):,} orders, {len(waits) - len(positive):,} "
                 "filled on arrival)")
    plt.show()
    display(waits.describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T)

## Markouts

The mid-price move after each fill, signed so that positive is good for the strategy: for a
buy, the mid at fill time plus the horizon minus the fill price; for a sell, the reverse.
Share-weighted means, in basis points of the fill price. A passive strategy that is being
adversely selected shows markouts that start positive (it earned half the spread) and decay
below zero as the horizon lengthens.

In [ ]:
HORIZONS = (1, 10, 60)
summary = report.markout_summary(run, HORIZONS)
display(summary)

if len(summary):
    columns = [f"markout_{h:g}s_bps" for h in HORIZONS]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for _, row in summary.iterrows():
        ax.plot([f"{h} s" for h in HORIZONS], [row[c] for c in columns], marker="o",
                label=f"{row['fills']} ({int(row['count']):,} fills)")
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_ylabel("mean markout (bps)")
    ax.set_title("Markout by horizon")
    ax.legend()
    plt.show()

## Queue position at arrival against fill outcome

Limit orders bucketed by how many shares were ahead of them when they went live.

Read this with care. All else equal a shorter queue should fill more often, but all else is
rarely equal: an order that finds almost nobody ahead of it has usually just joined a price
level that was created a moment ago, and such levels tend to vanish again (and the order is
cancelled) before anything trades there. On the reference day PassiveMaker's orders with the
longest queues ahead filled most often for exactly that reason. The table shows what happened;
it does not isolate the effect of queue position.

In [ ]:
queue = report.queue_vs_outcome(run)
display(queue)

if len(queue):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar([str(b) for b in queue["queue_bucket"]], queue["fill_probability"])
    ax.set_xlabel("shares ahead at arrival")
    ax.set_ylabel("fill probability")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    ax.set_title("Fill probability by queue position at arrival")
    plt.show()

## Risk

Rejections by reason code, and any kill-switch trip with the time and the total PnL at that
moment.

In [ ]:
rejects = report.rejects_by_reason(run)
if len(rejects):
    display(rejects)
    fig, ax = plt.subplots(figsize=(7, 0.5 + 0.4 * len(rejects)))
    ax.barh(rejects["reason"], rejects["count"])
    ax.invert_yaxis()
    ax.set_xlabel("rejected orders")
    ax.set_title("Risk rejections by reason")
    plt.show()
else:
    print("No orders were rejected.")

trips = report.kill_switch_events(run)
if len(trips):
    trips = trips.assign(time=trips["ts"].map(report.format_time))
    display(trips[["time", "reason", "total_pnl_usd"]])
else:
    print("The kill switch did not trip.")